# Task 10: MLP with one sensor vs all sensors as input

Runs the same code as on the PC (`src/utils/tuning/nn_tuning.py final`), only on a GPU. Eduard's MLP with its default settings
(2 layers x 256 units, dropout 0.5, learning rate 0.001), inputs speed + time of day + day of week, seeds 42 / 1 / 2:
- **one sensor:** each sensor is its own example, one model shared by all sensors (as in the other Step 2 models);
- **all sensors:** one example = the last 12 values of all 207 sensors, the model predicts all 207 at once.

Everything else is the same, so the difference shows whether seeing the other sensors (without the graph) helps or hurts.
Results go into `results_mlp.zip` (Output tab).

**Setup on Kaggle:** Add Input -> the dataset with the new `kaggle_upload.zip`; Settings -> Accelerator: **GPU T4 x2**, Internet: **on**.
Then **Save Version -> Save & Run All (Commit)**: it runs in the background, the browser can be closed.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

# the uploaded code + data + tuning database, wherever Kaggle put it; /kaggle/input is read-only, so work on a copy
WORK = '/kaggle/working/topic4'
marker = glob.glob('/kaggle/input/**/src/utils/tuning/nn_tuning.py', recursive=True)
if marker:                                   # Kaggle unpacked the zip
    shutil.copytree(marker[0].removesuffix('/src/utils/tuning/nn_tuning.py'), WORK, dirs_exist_ok=True)
else:                                        # the zip was kept as a file: unpack it here
    zipfile.ZipFile(glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]).extractall('/kaggle/working')
os.chdir(WORK)
print(sorted(os.listdir('.')), '| tuning database found:', os.path.exists('checkpoints/tuning/optuna.db'))

In [ ]:
# BasicTS without its pinned extras (its old `transformers` does not build on Kaggle and our code does not use it),
# plus the few small packages our code imports that Kaggle does not have
!pip install -q --no-deps basicts==1.1.0 easy-torch==1.3.3
!pip install -q easydict setproctitle optuna
!python -c "import basicts, easytorch, optuna; from src.utils.tuning.nn_tuning import MODELS; print('BasicTS', basicts.__version__, '| models:', list(MODELS))"
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'
print(torch.__version__, torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0))

## Train: one sensor (GPU 0) and all sensors (GPU 1) at the same time (logs: `checkpoints/final/MLP_*.log`)


In [ ]:
os.makedirs('checkpoints/final', exist_ok=True)
FINAL = [sys.executable, '-m', 'src.utils.tuning.nn_tuning', 'final', '--threads', '1',
         '--models', 'MLP', '--defaults', '--seeds', '42', '1', '2']
jobs = [  # (name, GPU, arguments)
    ('MLP_one_sensor', 0, ['--tag', 'one_sensor']),
    ('MLP_all_sensors', 1, ['--tag', 'all_sensors', '--params', '{"input_size": 207}'])]
n_gpus = torch.cuda.device_count()
procs = {}
for name, gpu, args in jobs:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % n_gpus))     # this process sees only this GPU
    procs[name] = subprocess.Popen(FINAL + args, env=env, stderr=subprocess.STDOUT,
                                   stdout=open(f'checkpoints/final/{name}.log', 'w'))
print({name: p.wait() for name, p in procs.items()})                  # exit code 0 = finished without errors


## Results and download


In [ ]:
import pandas as pd
rows = []
for path in sorted(glob.glob('checkpoints/final/MLP/*/*/test_metrics.json')):
    run = path.split('/')[3]
    m = json.load(open(path))
    rows.append({'run': run, **{f'test MAE {mins} min': m[f'horizon_{h}']['MAE'] for h, mins in [(3, 15), (6, 30), (12, 60)]}})
display(pd.DataFrame(rows).round(3))

with zipfile.ZipFile('/kaggle/working/results_mlp.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob('checkpoints/final/MLP/**', recursive=True) + glob.glob('checkpoints/final/MLP_*.log'):
        if os.path.isfile(path) and path.endswith(('test_metrics.json', 'cfg.json', '.log', '_best_val_MAE.pt')):
            z.write(path)
print(f"results_mlp.zip: {os.path.getsize('/kaggle/working/results_mlp.zip') / 1e6:.1f} MB -> Output tab -> download, "
      "then unzip it in the repo folder on the PC")
